In [2]:
import scanpy as sc
import omicverse as ov
import pandas as pd
ov.plot_set()


   ____            _     _    __                  
  / __ \____ ___  (_)___| |  / /__  _____________ 
 / / / / __ `__ \/ / ___/ | / / _ \/ ___/ ___/ _ \ 
/ /_/ / / / / / / / /__ | |/ /  __/ /  (__  )  __/ 
\____/_/ /_/ /_/_/\___/ |___/\___/_/  /____/\___/                                              

Version: 1.6.11, Tutorials: https://omicverse.readthedocs.io/
Dependency error: The 'phate>=1.0' distribution was not found and is required by the application


In [3]:
adata = sc.read("/home/lugli/spuccio/Projects/SP039/GBmap/Pombo2021_Part2.h5ad")

In [4]:
adata = adata[adata.obs['donor_id'].isin(["ND1", "ND2", "ND3", "ND4", "ND5", "ND6", "ND7", "ND8"])]

In [5]:
df_obs = pd.DataFrame(adata.obs)

In [6]:
del adata.obs

In [7]:
adata = adata.raw.to_adata()

In [8]:
adata

AnnData object with n_obs × n_vars = 27000 × 16024
    var: 'mt', 'n_cells', 'percent_cells', 'robust', 'means', 'variances', 'residual_variances', 'highly_variable_rank', 'highly_variable_features'
    uns: 'X_approximate_distribution', 'annotation_level_1_colors', 'annotation_level_2_colors', 'annotation_level_3_colors', 'batch_condition', 'default_embedding', 'donor_id_colors', 'hvg', 'leiden', 'leiden_colors', 'log1p', 'neighbors', 'pca', 'rank_genes_groups', 'scaled|original|cum_sum_eigenvalues', 'scaled|original|pca_var_ratios', 'schema_version', 'scsa_celltype_cellmarker_colors', 'scsa_celltype_panglaodb_colors', 'title', 'umap'
    obsm: 'X_harmony', 'X_pca', 'X_umap', 'scaled|original|X_pca'
    obsp: 'connectivities', 'distances'

In [9]:
#adata = adata.raw.to_adata()

In [10]:
X_counts_recovered, size_factors_sub=ov.pp.recover_counts(adata.X, 50*1e4, 50*1e5, log_base=None, 
                                                          chunk_size=10000)


100%|██████████| 7000/7000 [00:08<00:00, 797.94it/s]


In [11]:
adata.X = X_counts_recovered

In [12]:
annot = sc.queries.biomart_annotations(
    "hsapiens",
    ["external_gene_name","ensembl_gene_id", "start_position", "end_position", "chromosome_name",],
).set_index("external_gene_name")

In [13]:
annot

,ensembl_gene_id,start_position,end_position,chromosome_name
external_gene_name,,,,
MT-TF,ENSG00000210049,577,647,MT
MT-RNR1,ENSG00000211459,648,1601,MT
MT-TV,ENSG00000210077,1602,1670,MT
MT-RNR2,ENSG00000210082,1671,3229,MT
MT-TL1,ENSG00000209082,3230,3304,MT
...,...,...,...,...
SCMH1-DT,ENSG00000235358,41241772,41338644,1
LINC01740,ENSG00000228067,212467563,212556085,1
SLC44A3-AS1,ENSG00000293271,94585556,94855426,1


In [14]:
adata.var.columns

Index(['mt', 'n_cells', 'percent_cells', 'robust', 'means', 'variances',
       'residual_variances', 'highly_variable_rank',
       'highly_variable_features'],
      dtype='object')

In [15]:
adata.var = adata.var[['mt', 'n_cells', 'percent_cells', 'robust', 'means', 'variances',
       'residual_variances']]

In [16]:
adata.var 

,mt,n_cells,percent_cells,robust,means,variances,residual_variances
feature_name,,,,,,,
ZNF470-DT,False,175,0.218857,True,0.001165,0.000780,0.732630
ZNF367,False,819,1.024249,True,0.004887,0.002997,0.601441
SULT1B1,False,686,0.857918,True,0.004315,0.002897,0.705753
TRIM63,False,85,0.106302,True,0.000602,0.000388,0.678293
HDHD2,False,10365,12.962569,True,0.063325,0.034446,0.551284
...,...,...,...,...,...,...,...
MPIG6B,False,49,0.061280,True,0.000312,0.000186,0.618676
H4C15,False,80,0.100049,True,0.000509,0.000310,0.623489
LNCAROD,False,65,0.081290,True,0.000402,0.000230,0.600409


In [17]:
df_tmp = pd.merge(adata.var , annot, left_index=True, right_index=True, how='left')

In [18]:
df_tmp = df_tmp.reset_index().drop_duplicates(['feature_name']).set_index(['feature_name'])

In [19]:
adata.var = df_tmp

In [20]:
adata = adata[:,adata.var['chromosome_name'].isin(["1","2","3","4","5","6","7","8","9","10","11","12","13","14","15","16","17","18","19","20","21","22","X","Y","MT"])]

In [21]:
adata

View of AnnData object with n_obs × n_vars = 27000 × 13817
    var: 'mt', 'n_cells', 'percent_cells', 'robust', 'means', 'variances', 'residual_variances', 'ensembl_gene_id', 'start_position', 'end_position', 'chromosome_name'
    uns: 'X_approximate_distribution', 'annotation_level_1_colors', 'annotation_level_2_colors', 'annotation_level_3_colors', 'batch_condition', 'default_embedding', 'donor_id_colors', 'hvg', 'leiden', 'leiden_colors', 'log1p', 'neighbors', 'pca', 'rank_genes_groups', 'scaled|original|cum_sum_eigenvalues', 'scaled|original|pca_var_ratios', 'schema_version', 'scsa_celltype_cellmarker_colors', 'scsa_celltype_panglaodb_colors', 'title', 'umap'
    obsm: 'X_harmony', 'X_pca', 'X_umap', 'scaled|original|X_pca'
    obsp: 'connectivities', 'distances'

In [22]:
adata.obs['donor_id'] = df_obs['donor_id']

In [24]:
metadata_data = {
    'Author': ['Pombo2021'] * 8,
    'donor_id': ["ND1", "ND2", "ND3", "ND4", "ND5", "ND6", "ND7", "ND8"],
    'stage': ['Primary'] * 8,
    'assay': ['10x 5\' v1', '10x 5\' v1', '10x 5\' v1', '10x 5\' v1', '10x 5\' v1', 
              '10x 3\' v2', '10x 3\' v2', '10x 3\' v3'],
    'tissue': ['brain'] * 8,
    'Cells': ['CD45'] * 8,
    'Method': ['cell'] * 8
}

metadata_df = pd.DataFrame(metadata_data)

# Display the metadata DataFrame
print(metadata_df)

      Author donor_id    stage      assay tissue Cells Method
0  Pombo2021      ND1  Primary  10x 5' v1  brain  CD45   cell
1  Pombo2021      ND2  Primary  10x 5' v1  brain  CD45   cell
2  Pombo2021      ND3  Primary  10x 5' v1  brain  CD45   cell
3  Pombo2021      ND4  Primary  10x 5' v1  brain  CD45   cell
4  Pombo2021      ND5  Primary  10x 5' v1  brain  CD45   cell
5  Pombo2021      ND6  Primary  10x 3' v2  brain  CD45   cell
6  Pombo2021      ND7  Primary  10x 3' v2  brain  CD45   cell
7  Pombo2021      ND8  Primary  10x 3' v3  brain  CD45   cell


In [25]:
merged_obs_df = pd.merge(pd.DataFrame(adata.obs), metadata_df, left_on='donor_id', right_on='donor_id', how='left')

# Display the merged dataframe
print(merged_obs_df)

      donor_id     Author    stage      assay tissue Cells Method
0          ND1  Pombo2021  Primary  10x 5' v1  brain  CD45   cell
1          ND1  Pombo2021  Primary  10x 5' v1  brain  CD45   cell
2          ND1  Pombo2021  Primary  10x 5' v1  brain  CD45   cell
3          ND1  Pombo2021  Primary  10x 5' v1  brain  CD45   cell
4          ND1  Pombo2021  Primary  10x 5' v1  brain  CD45   cell
...        ...        ...      ...        ...    ...   ...    ...
26995      ND8  Pombo2021  Primary  10x 3' v3  brain  CD45   cell
26996      ND8  Pombo2021  Primary  10x 3' v3  brain  CD45   cell
26997      ND8  Pombo2021  Primary  10x 3' v3  brain  CD45   cell
26998      ND8  Pombo2021  Primary  10x 3' v3  brain  CD45   cell
26999      ND8  Pombo2021  Primary  10x 3' v3  brain  CD45   cell

[27000 rows x 7 columns]


In [26]:
df_obs = df_obs[['donor_id','n_genes','nUMIs','annotation_level_1', 'annotation_level_2','annotation_level_3','scsa_celltype_cellmarker', 'scsa_celltype_panglaodb','cell_type']]

In [27]:
df_obs

,donor_id,n_genes,nUMIs,annotation_level_1,annotation_level_2,annotation_level_3,scsa_celltype_cellmarker,scsa_celltype_panglaodb,cell_type
ND1_GTGCAGCGTACCGCTG-2-0,ND1,1149,1443.376343,Non-neoplastic,Myeloid,TAM-BDM,Microglial cell,Microglia,macrophage
ND1_TGGCTGGCAAGACACG-2-0,ND1,773,1192.182373,Non-neoplastic,Myeloid,TAM-MG,Microglial cell,Microglia,microglial cell
ND1_GACAGAGGTCTCACCT-2-0,ND1,1092,1392.278931,Non-neoplastic,Myeloid,TAM-MG,Microglial cell,Microglia,microglial cell
ND1_AACCGCGCAGTCGTGC-2-0,ND1,1279,1471.670044,Non-neoplastic,Myeloid,TAM-MG,Microglial cell,Macrophages,microglial cell
ND1_AACGTTGCAGCATACT-2-0,ND1,1582,1490.304810,Non-neoplastic,Myeloid,TAM-MG,Microglial cell,Macrophages,microglial cell
...,...,...,...,...,...,...,...,...,...
ND8_TTTGTTGCAATCCTAG-3-0,ND8,2276,2009.330933,Non-neoplastic,Myeloid,TAM-MG,Microglial cell,Microglia,microglial cell
ND8_TTTGTTGCACTGTGTA-3-0,ND8,3348,1836.801270,Non-neoplastic,Myeloid,TAM-MG,Microglial cell,Macrophages,microglial cell
ND8_TTTGTTGGTACAAAGT-3-0,ND8,2280,1691.894653,Non-neoplastic,Myeloid,TAM-MG,Microglial cell,Microglia,microglial cell
ND8_TTTGTTGGTGAGAGGG-3-0,ND8,1233,1506.083008,Non-neoplastic,Lymphoid,CD4/CD8,T cell,T Cells,mature T cell


In [28]:
merged_obs_df.index= df_obs.index

In [29]:
merged_obs_df.columns

Index(['donor_id', 'Author', 'stage', 'assay', 'tissue', 'Cells', 'Method'], dtype='object')

In [30]:
merged_obs_df = pd.merge(merged_obs_df, df_obs,right_index=True,left_index=True, how='left')

In [31]:
merged_obs_df.columns

Index(['donor_id_x', 'Author', 'stage', 'assay', 'tissue', 'Cells', 'Method',
       'donor_id_y', 'n_genes', 'nUMIs', 'annotation_level_1',
       'annotation_level_2', 'annotation_level_3', 'scsa_celltype_cellmarker',
       'scsa_celltype_panglaodb', 'cell_type'],
      dtype='object')

In [32]:
del merged_obs_df['donor_id_y']

In [33]:
merged_obs_df.columns = ['donor_id', 'Author', 'stage', 'assay', 'tissue', 'Cells', 'Method',
                         'n_genes', 'nUMIs', 'annotation_level_1',
       'annotation_level_2', 'annotation_level_3', 'scsa_celltype_cellmarker',
       'scsa_celltype_panglaodb', 'cell_type']

In [34]:
merged_obs_df.columns

Index(['donor_id', 'Author', 'stage', 'assay', 'tissue', 'Cells', 'Method',
       'n_genes', 'nUMIs', 'annotation_level_1', 'annotation_level_2',
       'annotation_level_3', 'scsa_celltype_cellmarker',
       'scsa_celltype_panglaodb', 'cell_type'],
      dtype='object')

In [35]:
adata.obs = merged_obs_df

In [36]:
ov.pp.score_genes_cell_cycle(adata,species='human')

calculating cell cycle phase
computing score 'S_score'
    finished: added
    'S_score', score of gene set (adata.obs).
    729 total control genes are used. (0:00:01)
computing score 'G2M_score'
    finished: added
    'G2M_score', score of gene set (adata.obs).
    726 total control genes are used. (0:00:01)
-->     'phase', cell cycle phase (adata.obs)


In [37]:
adata.write("/home/lugli/spuccio/Projects/SP039/GBmap/Pombo2021_Part3.h5ad")